# Music Annotation Project Notebook 65
## Page258 “SCORE + 27 parts” V7.8 Route-Provenance Audit

**Runtime:** CPU. Designed for Google Colab and Google Drive.

This notebook measures, for an exact Page258 annotation region:

- membership in the Notebook 07 corrected seed mask;
- membership in the frozen dilation-2 prepared mask;
- frozen V7.8 `preserve`, `remove`, and `ambiguous` routes;
- V8/59R2 eligibility under `prepared_mask & (observed < 200) & ~preserve`;
- exclusive and inclusive V7.8 support-channel attribution for preserved pixels;
- spatial overlays, exact pixel tables, and a curated upload package.

The preferred population is a manually reviewed binary stroke mask containing only **“SCORE + 27 parts”**. If no stroke mask is supplied, the notebook uses dark prepared-mask pixels inside a configurable ROI and labels the result as an ROI approximation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib


In [ ]:
# ============================================================
# Cell 2 - Configuration
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import sys
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
PAGE = 'Page258'

# Optional exact overrides. Leave as None for automatic resolution.
OBSERVED_PATH_OVERRIDE = None
NOTEBOOK07_MASK_PATH_OVERRIDE = None
NOTE_STROKE_MASK_PATH_OVERRIDE = None

# Tight default ROI based on the Notebook 64 Page258 top-right audit.
# Change this if the note is not fully enclosed.
NOTE_ROI = (1266, 15, 1452, 131)  # x0, y0, x1, y1

# Frozen project constants.
MASK_DILATION = 2
CROP_MARGIN = 48
GEOMETRY_THRESHOLD = 80
BAND_WIDTH = 4
DARK_THRESHOLD = 200
MIN_COMPONENT_AREA = 20

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook65_page258_route_provenance'
OUTPUT_ROOT = ROOT / 'notebook65_page258_route_provenance'
RUN = OUTPUT_ROOT / RUN_ID
TABLES = RUN / 'tables'
VISUALS = RUN / 'visuals'
AUDIT = RUN / 'audit'
UPLOAD = RUN / 'upload'

for folder in [RUN, TABLES, VISUALS, AUDIT, UPLOAD]:
    folder.mkdir(parents=True, exist_ok=False)

SUPPORTED = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}

print('Output:', RUN)


In [ ]:
# ============================================================
# Cell 3 - Helpers and automatic artefact resolution
# ============================================================

def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256_file(path):
    path = Path(path)
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1048576), b''):
            h.update(block)
    return h.hexdigest()


def array_hash(array):
    array = np.ascontiguousarray(array)
    h = hashlib.sha256()
    h.update(str(array.dtype).encode())
    h.update(str(array.shape).encode())
    h.update(array.tobytes())
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    temporary.replace(path)


def image_map(folder):
    folder = Path(folder)
    if not folder.exists():
        return {}
    return {
        path.stem: path
        for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED
    }


def latest_complete_run(root, required_relative_paths):
    root = Path(root)
    hits = []
    if root.exists():
        for candidate in root.rglob('*'):
            if candidate.is_dir() and all((candidate / item).exists() for item in required_relative_paths):
                hits.append(candidate)
    return max(hits, key=lambda path: path.stat().st_mtime) if hits else None


def resolve_observed(page):
    if OBSERVED_PATH_OVERRIDE is not None:
        path = Path(OBSERVED_PATH_OVERRIDE)
        require(path.is_file(), f'Observed override does not exist: {path}')
        return path

    direct_candidates = [
        ROOT / 'aligned' / f'{page}.png',
        ROOT / 'source' / 'annotated_only' / f'{page}.png',
    ]
    for path in direct_candidates:
        if path.is_file():
            return path

    patterns = [
        f'**/pages/{page}/observed.png',
        f'**/whole_pages/{page}/observed.png',
        f'**/{page}/observed.png',
    ]
    hits = []
    for pattern in patterns:
        hits.extend(ROOT.glob(pattern))
    hits = [path for path in hits if path.is_file()]
    require(hits, f'Could not resolve an observed image for {page}. Set OBSERVED_PATH_OVERRIDE.')
    return max(hits, key=lambda path: path.stat().st_mtime)


def resolve_seed_mask(page):
    if NOTEBOOK07_MASK_PATH_OVERRIDE is not None:
        path = Path(NOTEBOOK07_MASK_PATH_OVERRIDE)
        require(path.is_file(), f'Notebook 07 mask override does not exist: {path}')
        return path

    candidate_dirs = [
        ROOT / 'human_in_the_middle' / 'corrected_masks',
        ROOT / 'masks' / 'corrected',
        ROOT / 'production_pilot' / 'reviewed_masks',
    ]
    for folder in candidate_dirs:
        mapping = image_map(folder)
        if page in mapping:
            return mapping[page]

    hits = []
    for pattern in [
        f'**/{page}*corrected*mask*.png',
        f'**/{page}*authoritative*.png',
        f'**/corrected_masks/{page}.png',
        f'**/masks/corrected/{page}.png',
    ]:
        hits.extend(ROOT.glob(pattern))
    hits = [path for path in hits if path.is_file()]
    require(hits, f'Could not resolve the Notebook 07 corrected seed mask for {page}. Set NOTEBOOK07_MASK_PATH_OVERRIDE.')
    return max(hits, key=lambda path: path.stat().st_mtime)


def resolve_optional_stroke_mask():
    if NOTE_STROKE_MASK_PATH_OVERRIDE is None:
        return None
    path = Path(NOTE_STROKE_MASK_PATH_OVERRIDE)
    require(path.is_file(), f'Note stroke-mask override does not exist: {path}')
    return path

print('Helpers ready.')


In [ ]:
# ============================================================
# Cell 3a - Canonical frozen V7.8 reconstruction and routing functions
#
# Source lineage:
#   17_V7_8_Rebuilt_Independent_Data_Builder
#   09_Pilot_Proposed_Correction_Routing_Reviewer
# ============================================================


def binary(
    mask,
):
    return np.where(
        mask > 0,
        255,
        0,
    ).astype(
        np.uint8
    )


def prepare_mask(
    seed,
    dilation=MASK_DILATION,
):
    mask = binary(
        seed
    )

    if dilation > 0:

        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (
                2 * dilation + 1,
                2 * dilation + 1,
            ),
        )

        mask = cv2.dilate(
            mask,
            kernel,
        )

    return mask


def immutable(
    original,
    candidate,
    mask,
):
    output = original.copy()

    output[
        mask > 0
    ] = candidate[
        mask > 0
    ]

    require(
        np.array_equal(
            output[
                mask == 0
            ],
            original[
                mask == 0
            ],
        ),
        (
            'Outside-mask immutability failed.'
        ),
    )

    return output


def padded_components(
    mask,
):
    number_components, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            (
                mask > 0
            ).astype(
                np.uint8
            ),
            8,
        )
    )

    height, width = mask.shape

    boxes = []

    for component_id in range(
        1,
        number_components,
    ):

        x = int(
            stats[
                component_id,
                cv2.CC_STAT_LEFT,
            ]
        )

        y = int(
            stats[
                component_id,
                cv2.CC_STAT_TOP,
            ]
        )

        component_width = int(
            stats[
                component_id,
                cv2.CC_STAT_WIDTH,
            ]
        )

        component_height = int(
            stats[
                component_id,
                cv2.CC_STAT_HEIGHT,
            ]
        )

        area = int(
            stats[
                component_id,
                cv2.CC_STAT_AREA,
            ]
        )

        if area < 8:
            continue

        boxes.append(
            (
                max(
                    0,
                    x - CROP_MARGIN,
                ),
                max(
                    0,
                    y - CROP_MARGIN,
                ),
                min(
                    width,
                    x
                    + component_width
                    + CROP_MARGIN,
                ),
                min(
                    height,
                    y
                    + component_height
                    + CROP_MARGIN,
                ),
            )
        )


    changed = True

    while changed:

        changed = False

        merged = []

        while boxes:

            (
                ax0,
                ay0,
                ax1,
                ay1,
            ) = boxes.pop()

            remaining = []

            for (
                bx0,
                by0,
                bx1,
                by1,
            ) in boxes:

                separated = (
                    ax1 < bx0
                    or bx1 < ax0
                    or ay1 < by0
                    or by1 < ay0
                )

                if not separated:

                    ax0 = min(
                        ax0,
                        bx0,
                    )

                    ay0 = min(
                        ay0,
                        by0,
                    )

                    ax1 = max(
                        ax1,
                        bx1,
                    )

                    ay1 = max(
                        ay1,
                        by1,
                    )

                    changed = True

                else:

                    remaining.append(
                        (
                            bx0,
                            by0,
                            bx1,
                            by1,
                        )
                    )

            boxes = remaining

            merged.append(
                (
                    ax0,
                    ay0,
                    ax1,
                    ay1,
                )
            )

        boxes = merged

    return boxes


def telea_reconstruct(
    observed,
    mask,
):
    result = observed.copy()

    for (
        x0,
        y0,
        x1,
        y1,
    ) in padded_components(
        mask
    ):

        image_crop = observed[
            y0:y1,
            x0:x1,
        ]

        mask_crop = mask[
            y0:y1,
            x0:x1,
        ]

        candidate = cv2.inpaint(
            image_crop,
            mask_crop,
            3,
            cv2.INPAINT_TELEA,
        )

        local = result[
            y0:y1,
            x0:x1,
        ]

        local[
            mask_crop > 0
        ] = candidate[
            mask_crop > 0
        ]

        result[
            y0:y1,
            x0:x1,
        ] = local

    return immutable(
        observed,
        result,
        mask,
    )


def background_surface(
    observed,
    mask,
    sigma=45.0,
):
    visible = (
        (
            mask == 0
        )
        & (
            observed >= 200
        )
    )

    values = (
        observed.astype(
            np.float32
        )
        * visible
    )

    weights = visible.astype(
        np.float32
    )

    numerator = cv2.GaussianBlur(
        values,
        (
            0,
            0,
        ),
        sigma,
    )

    denominator = cv2.GaussianBlur(
        weights,
        (
            0,
            0,
        ),
        sigma,
    )

    surface = (
        numerator
        / np.maximum(
            denominator,
            1e-4,
        )
    )

    fallback = cv2.GaussianBlur(
        observed,
        (
            0,
            0,
        ),
        sigma,
    )

    return np.clip(
        np.where(
            denominator > 0.02,
            surface,
            fallback,
        ),
        0,
        255,
    ).astype(
        np.uint8
    )


def outside_dark(
    observed,
    mask,
):
    return (
        (
            observed < 190
        )
        & (
            mask == 0
        )
    )


def support_maps(
    observed,
    mask,
):
    dark = outside_dark(
        observed,
        mask,
    )


    horizontal_base = (
        cv2.morphologyEx(
            dark.astype(
                np.uint8
            )
            * 255,
            cv2.MORPH_OPEN,
            np.ones(
                (
                    1,
                    35,
                ),
                np.uint8,
            ),
        )
        > 0
    )


    vertical_base = (
        cv2.morphologyEx(
            dark.astype(
                np.uint8
            )
            * 255,
            cv2.MORPH_OPEN,
            np.ones(
                (
                    15,
                    1,
                ),
                np.uint8,
            ),
        )
        > 0
    )


    (
        number_components,
        labels,
        stats,
        _,
    ) = cv2.connectedComponentsWithStats(
        dark.astype(
            np.uint8
        ),
        8,
    )


    compact_base = np.zeros_like(
        dark,
        dtype=bool,
    )


    for component_id in range(
        1,
        number_components,
    ):

        component_width = int(
            stats[
                component_id,
                cv2.CC_STAT_WIDTH,
            ]
        )

        component_height = int(
            stats[
                component_id,
                cv2.CC_STAT_HEIGHT,
            ]
        )

        area = int(
            stats[
                component_id,
                cv2.CC_STAT_AREA,
            ]
        )


        if (
            3 <= area <= 650
            and component_width <= 40
            and component_height <= 40
        ):

            compact_base[
                labels == component_id
            ] = True


    support_kernel = np.ones(
        (
            2 * BAND_WIDTH + 1,
            2 * BAND_WIDTH + 1,
        ),
        np.uint8,
    )


    horizontal = (
        cv2.dilate(
            horizontal_base.astype(
                np.uint8
            ),
            support_kernel,
        )
        > 0
    )


    vertical = (
        cv2.dilate(
            vertical_base.astype(
                np.uint8
            ),
            support_kernel,
        )
        > 0
    )


    compact = (
        cv2.dilate(
            compact_base.astype(
                np.uint8
            ),
            support_kernel,
        )
        > 0
    )


    return (
        horizontal,
        vertical,
        compact,
    )


def geometry_t80(
    observed,
    mask,
    telea,
):
    paper = background_surface(
        observed,
        mask,
    )


    normalised = cv2.divide(
        observed,
        np.maximum(
            paper,
            1,
        ),
        scale=255,
    )


    (
        horizontal,
        vertical,
        compact,
    ) = support_maps(
        observed,
        mask,
    )


    inside_mask = (
        mask > 0
    )


    preserve = (
        inside_mask
        & (
            horizontal
            | vertical
            | compact
        )
        & (
            observed < DARK_THRESHOLD
        )
    )


    remove = (
        inside_mask
        & ~preserve
        & (
            observed
            >= GEOMETRY_THRESHOLD
        )
        & (
            normalised
            >= GEOMETRY_THRESHOLD
        )
    )


    ambiguous = (
        inside_mask
        & ~preserve
        & ~remove
    )


    result = observed.copy()

    result[
        remove
    ] = paper[
        remove
    ]

    result[
        ambiguous
    ] = telea[
        ambiguous
    ]

    result[
        preserve
    ] = observed[
        preserve
    ]


    result = immutable(
        observed,
        result,
        mask,
    )


    routes = {
        'preserve':
            preserve,

        'remove':
            remove,

        'ambiguous':
            ambiguous,

        'horizontal':
            horizontal,

        'vertical':
            vertical,

        'compact':
            compact,

        'paper':
            paper,

        'normalised':
            normalised,
    }


    return (
        result,
        routes,
    )

In [ ]:
# ============================================================
# Cell 4 - Resolve Page258 through observed page + Notebook 07 mask,
# then rebuild frozen V7.8 routes directly.
# ============================================================

OBSERVED_PATH = resolve_observed(
    PAGE
)

SEED_MASK_PATH = resolve_seed_mask(
    PAGE
)

NOTE_STROKE_MASK_PATH = (
    resolve_optional_stroke_mask()
)


observed = cv2.imread(
    str(
        OBSERVED_PATH
    ),
    cv2.IMREAD_GRAYSCALE,
)

seed_image = cv2.imread(
    str(
        SEED_MASK_PATH
    ),
    cv2.IMREAD_GRAYSCALE,
)


require(
    observed is not None,
    (
        f'Unreadable observed image: '
        f'{OBSERVED_PATH}'
    ),
)


require(
    seed_image is not None,
    (
        f'Unreadable Notebook 07 corrected mask: '
        f'{SEED_MASK_PATH}'
    ),
)


require(
    observed.shape
    == seed_image.shape,
    (
        'Observed/Notebook 07 mask shape mismatch: '
        f'{observed.shape} versus {seed_image.shape}'
    ),
)


seed_mask = (
    seed_image > 0
)


prepared_mask_u8 = prepare_mask(
    seed_image,
    MASK_DILATION,
)


prepared_mask = (
    prepared_mask_u8 > 0
)


telea = telea_reconstruct(
    observed,
    prepared_mask_u8,
)


geometry_t80_page, routes = geometry_t80(
    observed,
    prepared_mask_u8,
    telea,
)


preserve = routes[
    'preserve'
]

remove = routes[
    'remove'
]

ambiguous = routes[
    'ambiguous'
]

horizontal = routes[
    'horizontal'
]

vertical = routes[
    'vertical'
]

compact = routes[
    'compact'
]

paper = routes[
    'paper'
]

normalised = routes[
    'normalised'
]

In [ ]:
# ============================================================
# Cell 5 - Define the exact note population
# ============================================================

height, width = observed.shape
x0, y0, x1, y1 = [int(value) for value in NOTE_ROI]
x0 = max(0, x0)
y0 = max(0, y0)
x1 = min(width, x1)
y1 = min(height, y1)
require(x1 > x0 and y1 > y0, f'Invalid NOTE_ROI after clipping: {(x0, y0, x1, y1)}')

roi = np.zeros(observed.shape, dtype=bool)
roi[y0:y1, x0:x1] = True

if NOTE_STROKE_MASK_PATH is not None:
    note_stroke_image = cv2.imread(str(NOTE_STROKE_MASK_PATH), cv2.IMREAD_GRAYSCALE)
    require(note_stroke_image is not None, f'Unreadable note stroke mask: {NOTE_STROKE_MASK_PATH}')
    require(note_stroke_image.shape == observed.shape, 'Note stroke mask must have the same shape as Page258.')
    reviewed_note_strokes = note_stroke_image > 0
    note_population = reviewed_note_strokes & prepared_mask & (observed < DARK_THRESHOLD)
    population_definition = 'reviewed SCORE + 27 parts stroke mask intersected with prepared mask and observed < 200'
    population_is_exact_semantic_stroke_mask = True
else:
    reviewed_note_strokes = None
    note_population = roi & prepared_mask & (observed < DARK_THRESHOLD)
    population_definition = 'dark prepared-mask pixels inside NOTE_ROI; ROI approximation, not an exact semantic stroke mask'
    population_is_exact_semantic_stroke_mask = False

population_total = int(note_population.sum())
require(population_total > 0, 'The selected note population contains zero pixels.')

seed_note = note_population & seed_mask
prepared_not_seed = note_population & prepared_mask & ~seed_mask

population_record = {
    'page': PAGE,
    'roi': {'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1},
    'population_definition': population_definition,
    'population_is_exact_semantic_stroke_mask': population_is_exact_semantic_stroke_mask,
    'population_pixels': population_total,
    'inside_seed_mask_pixels': int(seed_note.sum()),
    'inside_seed_mask_fraction': float(seed_note.sum() / population_total),
    'added_only_by_dilation_pixels': int(prepared_not_seed.sum()),
    'added_only_by_dilation_fraction': float(prepared_not_seed.sum() / population_total),
}
atomic_json(AUDIT / 'note_population_identity.json', population_record)
print(json.dumps(population_record, indent=2))


In [ ]:
# ============================================================
# Cell 6 - Exact V7.8 routes and 59R2 eligibility
# ============================================================

route_masks = {
    'preserve': note_population & preserve,
    'remove': note_population & remove,
    'ambiguous': note_population & ambiguous,
}

route_union = np.logical_or.reduce(list(route_masks.values()))
route_overlap = note_population & (
    route_masks['preserve'].astype(np.uint8)
    + route_masks['remove'].astype(np.uint8)
    + route_masks['ambiguous'].astype(np.uint8)
    > 1
)

require(not (note_population & ~route_union).any(), 'One or more note-population pixels have no V7.8 route.')
require(not route_overlap.any(), 'One or more note-population pixels have overlapping V7.8 routes.')

eligible_59r2 = note_population & prepared_mask & (observed < DARK_THRESHOLD) & ~preserve

route_rows = []
for route_name, route_mask in route_masks.items():
    pixels = int(route_mask.sum())
    route_rows.append({
        'population': population_definition,
        'route': route_name,
        'pixels': pixels,
        'proportion': pixels / population_total,
    })

route_rows.append({
    'population': population_definition,
    'route': '59R2_eligible',
    'pixels': int(eligible_59r2.sum()),
    'proportion': float(eligible_59r2.sum() / population_total),
})

route_summary = pd.DataFrame(route_rows)
route_summary.to_csv(TABLES / 'Page258_SCORE_27_route_summary.csv', index=False)

display(route_summary)

# Eligibility is exactly the dark, non-preserved subset of this population.
require(
    np.array_equal(eligible_59r2, note_population & ~preserve),
    '59R2 eligibility identity failed inside the already-dark note population.',
)


In [ ]:
# ============================================================
# Cell 7 - Exclusive and inclusive support attribution
# ============================================================

preserved_note = note_population & preserve
preserved_total = int(preserved_note.sum())

support_categories = {
    'horizontal_only': preserved_note & horizontal & ~vertical & ~compact,
    'vertical_only': preserved_note & ~horizontal & vertical & ~compact,
    'compact_only': preserved_note & ~horizontal & ~vertical & compact,
    'horizontal_vertical': preserved_note & horizontal & vertical & ~compact,
    'horizontal_compact': preserved_note & horizontal & ~vertical & compact,
    'vertical_compact': preserved_note & ~horizontal & vertical & compact,
    'horizontal_vertical_compact': preserved_note & horizontal & vertical & compact,
}

support_partition_total = int(sum(mask.sum() for mask in support_categories.values()))
require(support_partition_total == preserved_total, 'Exclusive support categories do not partition preserved note pixels exactly.')

exclusive_rows = []
for category, mask in support_categories.items():
    pixels = int(mask.sum())
    exclusive_rows.append({
        'support_category': category,
        'pixels': pixels,
        'proportion_of_preserved': pixels / preserved_total if preserved_total else np.nan,
        'proportion_of_note_population': pixels / population_total,
    })
exclusive_support_summary = pd.DataFrame(exclusive_rows)
exclusive_support_summary.to_csv(TABLES / 'Page258_SCORE_27_exclusive_support_summary.csv', index=False)

inclusive_rows = []
for channel, channel_mask in {
    'horizontal': horizontal,
    'vertical': vertical,
    'compact': compact,
}.items():
    pixels = int((preserved_note & channel_mask).sum())
    inclusive_rows.append({
        'support_channel': channel,
        'pixels': pixels,
        'proportion_of_preserved': pixels / preserved_total if preserved_total else np.nan,
        'proportion_of_note_population': pixels / population_total,
    })
inclusive_support_summary = pd.DataFrame(inclusive_rows)
inclusive_support_summary.to_csv(TABLES / 'Page258_SCORE_27_inclusive_support_summary.csv', index=False)

print('Exclusive support attribution')
display(exclusive_support_summary)
print('Inclusive support attribution; proportions can sum above 1 because channels overlap')
display(inclusive_support_summary)


In [ ]:
# ============================================================
# Cell 8 - Pixel-level provenance table
# ============================================================

ys, xs = np.where(note_population)
pixel_rows = []

for y, x in zip(ys, xs):
    if preserve[y, x]:
        route = 'preserve'
    elif remove[y, x]:
        route = 'remove'
    elif ambiguous[y, x]:
        route = 'ambiguous'
    else:
        route = 'unrouted'

    support_names = []
    if horizontal[y, x]:
        support_names.append('horizontal')
    if vertical[y, x]:
        support_names.append('vertical')
    if compact[y, x]:
        support_names.append('compact')

    pixel_rows.append({
        'page': PAGE,
        'y': int(y),
        'x': int(x),
        'observed_value': int(observed[y, x]),
        'inside_notebook07_seed_mask': bool(seed_mask[y, x]),
        'inside_prepared_dilation2_mask': bool(prepared_mask[y, x]),
        'v78_route': route,
        'horizontal_support': bool(horizontal[y, x]),
        'vertical_support': bool(vertical[y, x]),
        'compact_support': bool(compact[y, x]),
        'support_channels': '|'.join(support_names) if support_names else 'none',
        'eligible_59r2': bool(eligible_59r2[y, x]),
    })

pixel_provenance = pd.DataFrame(pixel_rows)
pixel_provenance.to_csv(TABLES / 'Page258_SCORE_27_pixel_provenance.csv', index=False)
pixel_provenance.to_parquet(TABLES / 'Page258_SCORE_27_pixel_provenance.parquet', index=False)

require(len(pixel_provenance) == population_total, 'Pixel-provenance row count mismatch.')
require(not (pixel_provenance['v78_route'] == 'unrouted').any(), 'Unrouted pixel detected in provenance table.')

display(pixel_provenance.head())
print('Pixel rows:', len(pixel_provenance))


In [ ]:
# ============================================================
# Cell 9 - Route, support and eligibility overlays
# ============================================================

BGR = {
    'preserve': (255, 120, 40),
    'remove': (40, 190, 60),
    'ambiguous': (0, 165, 255),
    'eligible': (220, 0, 220),
    'horizontal': (255, 0, 0),
    'vertical': (0, 200, 255),
    'compact': (0, 0, 255),
}


def crop(array):
    return array[y0:y1, x0:x1]


def blend(gray, masks_and_colours, alpha=0.62):
    base = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    colour = base.copy()
    active = np.zeros(gray.shape, dtype=bool)
    for mask, bgr in masks_and_colours:
        colour[mask] = bgr
        active |= mask
    result = base.copy()
    mixed = cv2.addWeighted(base, 1 - alpha, colour, alpha, 0)
    result[active] = mixed[active]
    return result


def add_banner(image, title, items):
    h, w = image.shape[:2]
    banner = 88
    canvas = np.full((h + banner, w, 3), 255, dtype=np.uint8)
    canvas[banner:] = image
    cv2.putText(canvas, title, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (30, 30, 30), 1, cv2.LINE_AA)
    x = 10
    y = 62
    for label, colour in items:
        cv2.rectangle(canvas, (x, y - 15), (x + 18, y + 3), colour, -1)
        cv2.putText(canvas, label, (x + 24, y + 2), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (30, 30, 30), 1, cv2.LINE_AA)
        x += max(125, len(label) * 9 + 48)
    return canvas

observed_crop = crop(observed)

route_overlay = add_banner(
    blend(
        observed_crop,
        [
            (crop(route_masks['preserve']), BGR['preserve']),
            (crop(route_masks['remove']), BGR['remove']),
            (crop(route_masks['ambiguous']), BGR['ambiguous']),
        ],
    ),
    f'{PAGE} SCORE + 27 parts: V7.8 routes',
    [('preserve', BGR['preserve']), ('remove', BGR['remove']), ('ambiguous', BGR['ambiguous'])],
)

support_overlay = add_banner(
    blend(
        observed_crop,
        [
            (crop(preserved_note & horizontal), BGR['horizontal']),
            (crop(preserved_note & vertical), BGR['vertical']),
            (crop(preserved_note & compact), BGR['compact']),
        ],
    ),
    f'{PAGE} preserved note pixels: support channels',
    [('horizontal', BGR['horizontal']), ('vertical', BGR['vertical']), ('compact', BGR['compact'])],
)

eligibility_overlay = add_banner(
    blend(
        observed_crop,
        [
            (crop(preserved_note), BGR['preserve']),
            (crop(eligible_59r2), BGR['eligible']),
        ],
    ),
    f'{PAGE} SCORE + 27 parts: preserve versus 59R2 eligible',
    [('V7.8 preserve', BGR['preserve']), ('59R2 eligible', BGR['eligible'])],
)

population_image = np.where(crop(note_population), 255, 0).astype(np.uint8)
seed_image_crop = np.where(crop(seed_mask), 255, 0).astype(np.uint8)
prepared_image_crop = np.where(crop(prepared_mask), 255, 0).astype(np.uint8)

outputs = {
    'Page258_SCORE_27_note_population.png': population_image,
    'Page258_SCORE_27_seed_mask_crop.png': seed_image_crop,
    'Page258_SCORE_27_prepared_mask_crop.png': prepared_image_crop,
    'Page258_SCORE_27_route_overlay.png': route_overlay,
    'Page258_SCORE_27_support_overlay.png': support_overlay,
    'Page258_SCORE_27_eligibility_overlay.png': eligibility_overlay,
}

for filename, image in outputs.items():
    require(cv2.imwrite(str(VISUALS / filename), image), f'Failed to write {filename}')

print('Visuals:', VISUALS)


In [ ]:
# ============================================================
# Cell 10 - Summary, verdict and curated upload package
# ============================================================

summary = {
    'page': PAGE,
    'population_definition': population_definition,
    'population_is_exact_semantic_stroke_mask': population_is_exact_semantic_stroke_mask,
    'population_pixels': population_total,
    'route_counts': {row['route']: int(row['pixels']) for row in route_rows},
    'route_proportions': {row['route']: float(row['proportion']) for row in route_rows},
    'preserved_pixels': preserved_total,
    'exclusive_support_counts': {
        row['support_category']: int(row['pixels'])
        for row in exclusive_rows
    },
    'inclusive_support_counts': {
        row['support_channel']: int(row['pixels'])
        for row in inclusive_rows
    },
    'interpretation_boundary': (
        'If no manually reviewed stroke mask was supplied, results apply to dark prepared-mask pixels '
        'inside NOTE_ROI and may include nearby printed material. They must not be presented as exact '
        'semantic proportions for the handwritten note.'
    ),
}
atomic_json(RUN / 'Page258_SCORE_27_route_provenance_summary.json', summary)

checks = {
    'observed_and_seed_readable': observed is not None and seed_image is not None,
    'shape_identity': observed.shape == seed_image.shape,
    'prepared_mask_built_from_notebook07_seed': 
        bool(
            np.array_equal(
                prepared_mask_u8,
                prepare_mask(
                    seed_image,
                    MASK_DILATION,
                ),
            )
        ),
    'route_partition_exact': bool(np.all(route_count[prepared_mask] == 1)),
    'preserve_rule_exact':
        bool(
            np.array_equal(
                preserve,
                expected_preserve,
            )
        ),

    'remove_rule_exact':
        bool(
            np.array_equal(
                remove,
                expected_remove,
            )
        ),

    'ambiguous_rule_exact':
        bool(
            np.array_equal(
                ambiguous,
                expected_ambiguous,
            )
        ),

    'route_partition_exact':
        bool(
            np.all(
                route_count[
                    prepared_mask
                ]
                == 1
            )
        ),

    'outside_mask_immutable':
        bool(
            np.array_equal(
                geometry_t80_page[
                    ~prepared_mask
                ],
                observed[
                    ~prepared_mask
                ],
            )
        ),
    'note_population_nonempty': population_total > 0,
    'pixel_provenance_complete': len(pixel_provenance) == population_total,
    'support_partition_exact': support_partition_total == preserved_total,
}

first_failure = next((name for name, passed in checks.items() if not passed), None)
verdict = 'PASS' if first_failure is None else 'FAIL'

verdict_record = {
    'verdict': verdict,
    'first_failed_check': first_failure,
    'checks': checks,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'truth_used': False,
    'clean_page_used': False,
    'promotion_authority': False,
}
atomic_json(AUDIT / 'notebook65_verdict.json', verdict_record)
require(verdict == 'PASS', f'Notebook 65 failed: {first_failure}')

executive_lines = [
    'NOTEBOOK 65 - PAGE258 SCORE + 27 PARTS V7.8 ROUTE-PROVENANCE AUDIT',
    '',
    f'VERDICT: {verdict}',
    f'Population definition: {population_definition}',
    f'Population pixels: {population_total:,}',
    '',
    'ROUTES',
]
for row in route_rows:
    executive_lines.append(
        f"- {row['route']}: {int(row['pixels']):,} ({100 * float(row['proportion']):.4f}%)"
    )
executive_lines.extend(['', 'EXCLUSIVE SUPPORT AMONG PRESERVED PIXELS'])
for row in exclusive_rows:
    value = row['proportion_of_preserved']
    displayed = 'n/a' if pd.isna(value) else f'{100 * float(value):.4f}%'
    executive_lines.append(f"- {row['support_category']}: {int(row['pixels']):,} ({displayed})")
executive_lines.extend([
    '',
    'INTERPRETATION',
    'Notebook 07 seed-mask membership, prepared-mask membership, V7.8 routes, support channels and 59R2 eligibility are separate states.',
    '59R2 eligibility is the dark, prepared-mask, non-preserved population.',
    'If the ROI approximation was used, create a manually reviewed note-stroke mask before treating the percentages as exact for the handwriting itself.',
])
(RUN / 'executive_summary.txt').write_text('\n'.join(executive_lines) + '\n')
print('\n'.join(executive_lines))

curated = [
    RUN / 'executive_summary.txt',
    RUN / 'Page258_SCORE_27_route_provenance_summary.json',
    AUDIT / 'resolved_input_identity.json',
    AUDIT / 'note_population_identity.json',
    AUDIT / 'notebook65_verdict.json',
    TABLES / 'Page258_SCORE_27_route_summary.csv',
    TABLES / 'Page258_SCORE_27_exclusive_support_summary.csv',
    TABLES / 'Page258_SCORE_27_inclusive_support_summary.csv',
    TABLES / 'Page258_SCORE_27_pixel_provenance.csv',
    TABLES / 'Page258_SCORE_27_pixel_provenance.parquet',
] + sorted(VISUALS.glob('*.png'))

for source in curated:
    require(source.is_file(), f'Missing curated artefact: {source}')
    relative = source.relative_to(RUN)
    destination = UPLOAD / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    require(sha256_file(source) == sha256_file(destination), f'Copy hash mismatch: {relative}')

(UPLOAD / 'README.txt').write_text(
    'Notebook 65 Page258 SCORE + 27 parts V7.8 route-provenance package.\n'
    'No clean truth is used.\n'
    'If no exact reviewed note-stroke mask was supplied, percentages describe dark prepared-mask pixels inside the configured ROI.\n'
)

file_list = sorted(
    str(path.relative_to(UPLOAD))
    for path in UPLOAD.rglob('*')
    if path.is_file() and path.name not in {'upload_file_list.txt', 'upload_manifest.csv'}
)
(UPLOAD / 'upload_file_list.txt').write_text('\n'.join(file_list) + '\n')

manifest_rows = []
for path in sorted(UPLOAD.rglob('*')):
    if path.is_file() and path.name != 'upload_manifest.csv':
        manifest_rows.append({
            'relative_path': str(path.relative_to(UPLOAD)),
            'bytes': int(path.stat().st_size),
            'sha256': sha256_file(path),
        })
pd.DataFrame(manifest_rows).drop_duplicates('relative_path').sort_values('relative_path').to_csv(
    UPLOAD / 'upload_manifest.csv', index=False
)

zip_path = RUN / 'Notebook65_Page258_SCORE_27_V78_Route_Provenance_upload_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(UPLOAD.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(UPLOAD))
with zipfile.ZipFile(zip_path, 'r') as archive:
    require(archive.testzip() is None, 'ZIP integrity test failed.')

package_identity = {
    'zip_path': str(zip_path),
    'zip_sha256': sha256_file(zip_path),
    'zip_bytes': int(zip_path.stat().st_size),
    'upload_files': len(manifest_rows),
}
atomic_json(RUN / 'upload_package_identity.json', package_identity)

print('Upload ZIP:', zip_path)
print('Upload ZIP SHA-256:', package_identity['zip_sha256'])


## Interpretation

Use `Page258_SCORE_27_route_summary.csv` for the route proportions and `Page258_SCORE_27_exclusive_support_summary.csv` for mutually exclusive support combinations.

If `population_is_exact_semantic_stroke_mask` is `false`, the notebook has measured a tight ROI approximation. For an exact statement about the handwriting alone, create a binary full-page mask containing only the reviewed “SCORE + 27 parts” strokes and set `NOTE_STROKE_MASK_PATH_OVERRIDE`.
